[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anicka-net/nla-at-home/blob/main/notebooks/03_roundtrip_faithfulness.ipynb)

# 03 · Round-Trip & Faithfulness

### HAAISS workshop — core notebook 3 of 4 (code-along)

So far we went **activation → English** (the AV, verbalizer). There is a second adapter that goes back **English → activation** (the AR, *reconstructor*). Chaining them gives a **round-trip**:

```
vector  --AV-->  caption  --AR-->  vector'
```
A faithful caption should help reconstruct the input-specific part of the vector. Raw cosine is only a first check because shared activation means can make wrong reconstructions look good; the useful test below compares centered reconstructions against distractors.

## Setup — same base, two adapters
The clever part: **AV and AR are both LoRA adapters on the same Qwen base.** We load the base once, attach both, and hot-swap. That's why the round-trip fits a free T4.

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import warnings

# bitsandbytes 0.47 calls a PyTorch helper scheduled for removal. This is a
# dependency deprecation, not a problem with the quantized model or notebook.
warnings.filterwarnings("ignore", message=r"_check_is_size will be removed.*",
                        category=FutureWarning,
                        module=r"bitsandbytes\.backends\.cuda\.ops")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE       = "Qwen/Qwen2.5-7B-Instruct"          # the model whose mind we read
AV_ADAPTER = "anicka/nla-qwen2.5-7b-universal-av-grpo"    # the "verbalizer" (activation -> English)
LAYER      = 20                                   # chosen layer; both adapters are universal
DEPTH_PCT  = 71                                   # nearest trained depth tag for layer 20
INJECT_CHAR  = "\u320e"                          # the placeholder token we overwrite: ㈎
INJECT_SCALE = 150.0                              # we normalize the activation's L2 norm TO this

/home/anicka/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
AR_ADAPTER = "anicka/nla-qwen2.5-7b-universal-ar"   # English -> activation

In [4]:
device = "cuda"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

# 4-bit so a 7B model + adapters fit a free-Colab T4 (16 GB). fp16 compute:
# the GRPO-sharpened adapter is numerically sensitive, and fp16 on CUDA is a
# tested-safe path (bf16 on Apple MPS collapses it; not our case here).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tok  = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0})
model = PeftModel.from_pretrained(base, AV_ADAPTER).eval()   # adapter name = "default"

inject_id = tok.encode(INJECT_CHAR, add_special_tokens=False)
assert len(inject_id) == 1, f"injection char must be ONE token, got {inject_id}"
inject_id = inject_id[0]
print("loaded — base + AV adapter on", next(model.parameters()).device)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/339 [00:08<48:42,  8.65s/it]

Loading weights:   1%|          | 2/339 [00:17<47:51,  8.52s/it]

Loading weights:   1%|          | 4/339 [00:18<19:37,  3.51s/it]

Loading weights:   1%|▏         | 5/339 [00:19<15:19,  2.75s/it]

...

Loading weights:  97%|█████████▋| 330/339 [01:49<00:04,  2.24it/s]

Loading weights:  99%|█████████▊| 334/339 [01:49<00:01,  3.78it/s]

Loading weights:  99%|█████████▉| 336/339 [01:49<00:00,  4.48it/s]

Loading weights: 100%|██████████| 339/339 [01:49<00:00,  3.09it/s]

loaded — base + AV adapter on cuda:0


In [5]:
def get_layers(m):
    """Reach the transformer block list through the PEFT + CausalLM wrappers."""
    b = m.base_model.model if hasattr(m, "base_model") else m
    inner = b.model if hasattr(b, "model") else b
    return inner.layers

def read_activation(prompt, layer=LAYER, max_new_tokens=128):
    """Grab the clean base-model residual at the last prompt token."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)

    grab = {}
    def hook(mod, inpt, out):
        h = out[0] if isinstance(out, tuple) else out
        if "h" not in grab:                 # FIRST forward pass only — otherwise
            grab["h"] = h[:, -1, :].detach() # every generated token overwrites it
    handle = get_layers(model)[layer].register_forward_hook(hook)
    try:
        with model.disable_adapter(), torch.no_grad():
            out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
    finally:
        handle.remove()
    reply = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return grab["h"].squeeze(0), reply

def normalize_to(v, scale=INJECT_SCALE):
    """Rescale v so its L2 norm equals `scale`. NOT v * scale — see notebook 02."""
    n = v.float().norm().clamp_min(1e-12)
    return v * (scale / n)

def av_prompt(depth_pct):
    return (
        "You are a meticulous AI researcher conducting an important investigation "
        "into activation vectors from a language model. Your overall task is to "
        "describe the semantic content of that activation vector.\n\n"
        "We will pass the vector enclosed in <concept> tags into your context, "
        "along with the network depth where it was extracted. "
        "You must then produce an explanation for the vector, enclosed within "
        "<explanation> tags. The explanation consists of 2-3 text snippets "
        "describing that vector.\n\n"
        f"Here is the vector from depth {depth_pct}% of the network:\n\n"
        f"<concept>{INJECT_CHAR}</concept>\n\n"
        "Please provide an explanation.\n\n"
        "<explanation>")

def describe(activation, depth=DEPTH_PCT, max_new_tokens=120, scale_fn=normalize_to):
    """The whole NLA read: build the prompt, overwrite the placeholder token's
    embedding with the (rescaled) activation, let the model narrate."""
    chat = tok.apply_chat_template([{"role": "user", "content": av_prompt(depth)}],
                                   tokenize=False, add_generation_prompt=True)
    ids = tok.encode(chat, add_special_tokens=False)  # match training: chat-wrapped, no BOS
    pos = ids.index(inject_id)
    input_ids = torch.tensor([ids], device=device)
    emb = model.get_input_embeddings()(input_ids).clone()
    emb[0, pos, :] = scale_fn(activation.to(emb.dtype))
    attn = torch.ones((1, len(ids)), device=device, dtype=torch.long)
    with torch.no_grad():
        out = model.generate(input_ids=input_ids, inputs_embeds=emb, attention_mask=attn,
                             max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    seq = out[0]
    gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq  # embeds path returns new-only
    return tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()

In [6]:
# attach the reconstructor alongside the verbalizer on the SAME base model
model.load_adapter(AR_ADAPTER, adapter_name="ar")   # AV is "default"
print("adapters:", list(model.peft_config.keys()))

adapters: ['default', 'ar']


## The reconstructor

The AR adapter is trained so that, when it reads a caption, the residual stream at **layer 20** *becomes* the activation being described. No extra head: the reconstruction is literally the hidden state at layer 20, last token.

In [7]:
import torch.nn.functional as F
AR_TEMPLATE = ("Summary of the following text from depth {depth}%: "
               "<text>{explanation}</text> <summary>\u320e")  # trailing ㈎ = readout position

def reconstruct(description):
    model.set_adapter("ar")
    ids = tok.encode(AR_TEMPLATE.format(explanation=description, depth=DEPTH_PCT),
                     add_special_tokens=False)
    try:
        with torch.no_grad():
            out = model(input_ids=torch.tensor([ids], device=device),
                        output_hidden_states=True, use_cache=False)
    finally:
        model.set_adapter("default")                # always switch back to the AV
    return out.hidden_states[LAYER + 1][0, -1].float().cpu()

def cos(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

## The round-trip

In [8]:
prompt = "Explain how a hash map handles collisions."
model.set_adapter("default")
activation, reply = read_activation(prompt)  # helper disables all adapters internally
caption = describe(activation)
back    = reconstruct(caption)

print("caption      :", caption)
print("round-trip cos:", round(cos(activation.float().cpu(), back), 3))

caption      : - Hash table: data structure for key-value storage
- "how does hash table works" frame: mechanism-explanation response
- "in terms of hashing algorithm" constraint: technical precision (hashing function, collision resolution)
- "in computer science" context: formal, neutral register
- Response strategy: step-by-step procedural explanation (hashing, storing, retrieving keys and values) with a focus on collision handling (chaining or probing)
round-trip cos: 0.952


> **Anchor:** the single raw cosine printed here can be around 0.9 because Qwen reconstructions share a large mean component. It is not the benchmark. On the published 286-text clean holdout, this AV's centered round-trip mean is **0.628**; the next cells show why centering and distractors are necessary.

## Faithfulness as a gap — and a trap

Now the payoff. Take the *real* caption and a deliberately *wrong* one, reconstruct both, and compare cosine to the true activation. First, the **naive** way — watch it fail:

In [9]:
wrong = "- Recipe for Thai green curry with coconut milk and basil\n"\
        "- Step-by-step cooking instructions for dinner"

c_real  = cos(activation.float().cpu(), reconstruct(caption))
c_wrong = cos(activation.float().cpu(), reconstruct(wrong))
print(f"cos(real caption)  = {c_real:.3f}")
print(f"cos(wrong caption) = {c_wrong:.3f}")
print(f"raw gap = {c_real - c_wrong:+.3f}   ...the wrong caption can still score implausibly high. Why?")

cos(real caption)  = 0.952
cos(wrong caption) = 0.792
raw gap = +0.160   ...the wrong caption can still score implausibly high. Why?


The wrong caption can still receive a surprisingly high raw cosine because AR reconstructions share a large mean component. A single wild negative may still rank below the true caption, but that does not calibrate faithfulness. We therefore reconstruct a lineup of distractors, subtract their mean, and compare the input-specific deviations.

In [10]:
# === EDIT THE DISTRACTORS to probe the detector ===
DISTRACTORS = [
    wrong,  # the curry recipe from above
    "- Legal contract clause about liability limitation active\n- Formal register, defined terms",
    "- Football match commentary, goal celebration active\n- Present-tense excited sports narration",
    "- Romantic poetry about moonlight and longing\n- Metaphor-dense lyrical register",
    "- Python exception traceback analysis active\n- Debugging context, error-message vocabulary",
]
recons = [reconstruct(d) for d in DISTRACTORS]
mean_recon = torch.stack(recons).mean(0)

a_dev    = activation.float().cpu() - mean_recon
true_dev = reconstruct(caption) - mean_recon
scores   = {"TRUE caption": cos(a_dev, true_dev)}
for d, r in zip(DISTRACTORS, recons):
    scores[d.split(chr(10))[0][:48]] = cos(a_dev, r - mean_recon)

ranked = sorted(scores.items(), key=lambda kv: -kv[1])
for name, s in ranked:
    mark = " <-- the vector votes for this one" if name == "TRUE caption" else ""
    print(f"{s:+.3f}  {name}{mark}")

centered_gap = scores["TRUE caption"] - max(v for k, v in scores.items() if k != "TRUE caption")
print(f"\ncentered gap = {centered_gap:+.3f}   (positive => the vector prefers the truth)")

+0.816  TRUE caption <-- the vector votes for this one
+0.159  - Python exception traceback analysis active
+0.087  - Recipe for Thai green curry with coconut milk 
+0.068  - Legal contract clause about liability limitati
-0.181  - Football match commentary, goal celebration ac
-0.183  - Romantic poetry about moonlight and longing

centered gap = +0.657   (positive => the vector prefers the truth)


Try harder distractors — a *near-miss* (same domain, wrong detail) vs a *wild miss* (unrelated topic). The centered round-trip gap should shrink for near-misses: the detector is graded, not binary. That gradient makes it usable as a reward or ranking signal. The separate **compass reranker** introduced in the slides uses a linear activation-to-text predictor rather than the AR.

---
### ✅ Self-check
Expected: both adapters load; the wrong caption still has a high raw cosine; the **TRUE caption ranks #1** in the centered comparison with a clearly positive gap. Exact values vary with quantization and decoding. If the true caption does not win, check `LAYER+1` indexing and adapter switching.

In [11]:
assert ranked[0][0] == "TRUE caption", "centered ranking failed — see self-check note"
assert centered_gap > 0.03, f"centered gap suspiciously small: {centered_gap:+.3f}"
print(f"self-check: TRUE caption ranks #1, centered gap {centered_gap:+.3f} ✓")

self-check: TRUE caption ranks #1, centered gap +0.657 ✓
